# JHS Integrated Science Flashcards: Fine-tuning Gemma 3 1B with LoRA

**Capstone project, Google DeepMind AI Research Foundations (course 08, Conversation pathway).**

**What.** This notebook fine-tunes Gemma 3 1B with LoRA (Keras / `keras_hub`, JAX backend) so that it answers a Ghanaian Junior High School (JHS) Integrated Science revision question with a short flashcard:

```
Category: Integrated Science
<2 to 4 plain-English sentences>
```

**Who it is for.** JHS students revising for the BECE, and the teachers who support them, in schools with little or no internet access. The long-term goal is to run the small model offline on a low-cost classroom device.

**Why.** A general-purpose model gives long, unfocused or off-syllabus answers. Fine-tuning on hand-written, curriculum-aligned cards should make answers short, consistently formatted and pitched at the right level.

## How to run

1. **Runtime > Change runtime type > T4 GPU**, then **Save**.
2. **Kaggle access** (same as the course labs): accept the Gemma 3 licence at https://www.kaggle.com/models/keras/gemma3/keras/ and add two Colab secrets (key icon in the left sidebar), `KAGGLE_USERNAME` and `KAGGLE_KEY`, with **Notebook access** switched on.
3. **Data**: have `jhs_flashcards.jsonl` ready. By default the notebook asks you to upload it (section 2). If you prefer Google Drive, put it at `MyDrive/jhs_flashcards.jsonl` and set `USE_DRIVE = True` in section 2.
4. **Runtime > Run all**. Training plus evaluation takes roughly 20 to 40 minutes on a T4.

If you run out of GPU memory, see section 12 at the bottom.

## 1. Setup

Install the course package (it pins the `keras`, `keras_hub` and `jax` versions used in the course labs), set the Keras backend and XLA memory flags **before** importing Keras, and fix the random seed to 812.

In [ ]:
# Install the custom package for this course (same command as the course labs).
# This pins keras, keras_hub and jax to the versions the labs were tested with.
!pip install -q "git+https://github.com/google-deepmind/ai-foundations.git@main"

In [ ]:
import os

# These must be set before Keras or JAX are imported.
os.environ["KERAS_BACKEND"] = "jax"  # Set the Keras backend to JAX.
# Avoids memory fragmentation on JAX backend (course labs 5.6 and 8.9).
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
# Disable command buffer pre-allocation and XLA dot merging, which on T4 GPUs
# keeps extra float32 activations in memory during training (course lab 7.5).
os.environ["XLA_FLAGS"] = (
    "--xla_gpu_enable_command_buffer= --xla_gpu_dot_merger_threshold_mb=0"
)

# Standard library imports.
import collections  # Counting topics and tokens.
import json  # Reading and writing JSONL files.
import random  # Seeding Python randomness.
import re  # Regular expressions for text processing.
import shutil  # Copying and zipping output files.
import unicodedata  # Unicode character categories for cleaning.
from textwrap import fill  # Wrapping long answers for printing.
from typing import Any

# Third-party data and utility libraries.
import matplotlib.pyplot as plt  # Plotting the training curves.
import numpy as np  # Numerical computing.
import pandas as pd  # Data manipulation and analysis.
from IPython.display import display  # Showing DataFrames.
from sklearn.model_selection import train_test_split  # Train/test split.

# Deep learning libraries.
import jax  # JAX backend (used here to check the GPU).
import keras  # High-level neural networks API.
import keras_hub  # Gemma 3 implementation in Keras.

# Google Colab specific.
from google.colab import drive  # Access to Google Drive.
from google.colab import files  # Upload and download files.
from google.colab import userdata  # Access to Colab secrets.

SEED = 812
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)  # Replicate randomness in Keras.

print("keras:", keras.__version__)
print("keras_hub:", keras_hub.__version__)
print("jax:", jax.__version__)
print("devices:", jax.devices())
if jax.default_backend() != "gpu":
    print(
        "WARNING: no GPU found. Use Runtime > Change runtime type > T4 GPU, "
        "then Runtime > Restart session and run all cells again."
    )

In [ ]:
# Load the Kaggle credentials from Colab secrets so keras_hub can download
# the Gemma 3 weights.
try:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception as error:  # SecretNotFoundError or NotebookAccessError.
    raise RuntimeError(
        "Could not read the Kaggle secrets. Add KAGGLE_USERNAME and KAGGLE_KEY "
        "in the Colab Secrets panel (key icon) and switch on 'Notebook access'."
    ) from error
print("Kaggle credentials loaded for user:", os.environ["KAGGLE_USERNAME"])

## 2. Load the data

Each line of `jhs_flashcards.jsonl` has a `prompt` (the question), a `response` (the flashcard, starting with `Category: Integrated Science`) and a `meta` dictionary with the topic.

* **Default (`USE_DRIVE = False`)**: a file picker appears. Choose `jhs_flashcards.jsonl` from your computer. If the file was already uploaded in this session, the picker is skipped.
* **Drive (`USE_DRIVE = True`)**: the notebook mounts Google Drive and reads `/content/drive/MyDrive/jhs_flashcards.jsonl`. Outputs are then also copied to `MyDrive/jhs_capstone_outputs/`.

In [ ]:
USE_DRIVE = False  # Set to True to read the dataset from Google Drive.
DATA_FILENAME = "jhs_flashcards.jsonl"
DRIVE_DATA_PATH = f"/content/drive/MyDrive/{DATA_FILENAME}"
LOCAL_DATA_PATH = f"/content/{DATA_FILENAME}"

OUTPUT_DIR = "/content/jhs_capstone_outputs"  # Everything this notebook saves.
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/jhs_capstone_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def get_data_path() -> str:
    """Returns the path of the dataset, uploading or mounting Drive if needed.

    Returns:
      The local path of the JSONL dataset.
    """
    if USE_DRIVE:
        drive.mount("/content/drive")
        if not os.path.exists(DRIVE_DATA_PATH):
            raise FileNotFoundError(
                f"{DRIVE_DATA_PATH} not found. Upload {DATA_FILENAME} to the "
                "top level of My Drive, or set USE_DRIVE = False."
            )
        return DRIVE_DATA_PATH

    if os.path.exists(LOCAL_DATA_PATH):
        print(f"Using the file already uploaded at {LOCAL_DATA_PATH}.")
        return LOCAL_DATA_PATH

    print(f"Choose {DATA_FILENAME} in the file picker below.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No file was uploaded.")
    # Save under a fixed name, even if Colab renamed the upload.
    name, content = next(iter(uploaded.items()))
    with open(LOCAL_DATA_PATH, "wb") as f:
        f.write(content)
    print(f"Saved upload '{name}' to {LOCAL_DATA_PATH}.")
    return LOCAL_DATA_PATH


data_path = get_data_path()
df = pd.read_json(data_path, lines=True)

missing = {"prompt", "response"} - set(df.columns)
if missing:
    raise ValueError(f"Dataset is missing columns: {missing}")

print("Columns:", df.columns.tolist())
print("Number of rows:", len(df))
if "meta" in df.columns:
    topics = collections.Counter(
        m.get("topic", "?") for m in df["meta"] if isinstance(m, dict)
    )
    print("Distinct topics:", len(topics))
display(df.head())

print("\nSample card:")
print("Prompt:", df.loc[0, "prompt"])
print(fill(df.loc[0, "response"], replace_whitespace=False))

## 3. Clean the data

`clean_html` and `clean_unicode` follow course lab 8.8. The data is hand-written, so little should change. One small adaptation: the course's `clean_unicode` deletes every symbol, which would turn `°C` into `C`, so this version keeps a short allow-list (`extra_keep`) of science symbols.

The original response is kept in `response_original`. Rows with an empty prompt or response, and duplicate prompts (ignoring case and spacing), are dropped.

In [ ]:
def clean_html(text: str) -> str:
    """Strips basic HTML markup and common entities from a string.

    Args:
      text: The text string that may contain HTML tags or entities.

    Returns:
      A cleaned string with tags stripped and the entities '&nbsp;', '&amp;',
      '&lt;' and '&gt;' converted to ' ', '&', '<' and '>'.
    """
    text = re.sub(r"<.*?>", "", text)  # Remove HTML tags.
    text = re.sub("&nbsp;", " ", text)  # Replace non-breaking space.
    text = re.sub("&amp;", "&", text)
    text = re.sub("&lt;", "<", text)
    text = re.sub("&gt;", ">", text)
    return text


def clean_unicode(text: str, extra_keep: str = "°%+=") -> str:
    """Removes non-text unicode characters (emojis, symbols) from a string.

    Args:
      text: The original text which may contain special characters.
      extra_keep: Characters to keep even though they are symbols, for example
        the degree sign in "37 °C".

    Returns:
      The input text with emojis and other non-text symbols removed.
    """
    categories_to_keep = ("L", "N", "P")  # Letters, numbers, punctuation.
    keep = []
    for ch in text:
        if (
            ch.isspace()
            or ch in extra_keep
            or unicodedata.category(ch).startswith(categories_to_keep)
        ):
            keep.append(ch)
    return "".join(keep)


def normalise_prompt(text: str) -> str:
    """Returns a lower-case, single-spaced key used to find duplicate prompts."""
    return " ".join(text.lower().split())


rows_before = len(df)
df["response_original"] = df["response"].astype(str)
df["prompt"] = df["prompt"].astype(str).map(clean_html).map(clean_unicode)
df["prompt"] = df["prompt"].str.strip()
df["response_clean"] = (
    df["response"].astype(str).map(clean_html).map(clean_unicode).str.strip()
)

changed = df[df["response_clean"] != df["response_original"].str.strip()]
print(f"Responses changed by cleaning: {len(changed)}")
for _, row in changed.head(3).iterrows():
    print("  BEFORE:", repr(row["response_original"][:120]))
    print("  AFTER: ", repr(row["response_clean"][:120]))

# Drop empty prompts or responses.
empty_mask = (df["prompt"] == "") | (df["response_clean"] == "")
df = df.loc[~empty_mask].reset_index(drop=True)

# Drop duplicate prompts, keeping the first occurrence.
dup_mask = df["prompt"].map(normalise_prompt).duplicated(keep="first")
df = df.loc[~dup_mask].reset_index(drop=True)

# Replace the original response with the cleaned version (as in lab 8.8).
df = df.drop(columns=["response"]).rename(columns={"response_clean": "response"})

not_formatted = (~df["response"].str.startswith("Category:")).sum()
print(f"\nRows before cleaning: {rows_before}")
print(f"Dropped empty rows: {int(empty_mask.sum())}")
print(f"Dropped duplicate prompts: {int(dup_mask.sum())}")
print(f"Rows after cleaning: {len(df)}")
print(f"Responses not starting with 'Category:': {int(not_formatted)}")
display(df.head())

## 4. Split into train and test sets

80/20 split with `random_state=812` so the split is the same every run. Both splits are saved as JSONL with `input` / `output` keys, exactly like lab 8.8, and then reloaded with `to_dict("records")` like lab 8.9.

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=SEED)


def to_instruction_record(row: pd.Series) -> dict[str, str]:
    """Transforms a single row of the dataset into instruction-tuning format.

    Args:
      row: A single row from a pandas DataFrame.

    Returns:
      A dictionary with "input" and "output" as keys.
    """
    return {"input": row["prompt"], "output": row["response"]}


def save_jsonl(records: list[dict[str, str]], path: str) -> None:
    """Writes a list of dictionaries to a JSONL file."""
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} records to {path}")


TRAIN_PATH = f"{OUTPUT_DIR}/conversational_training_data.jsonl"
TEST_PATH = f"{OUTPUT_DIR}/conversational_test_data.jsonl"
save_jsonl([to_instruction_record(r) for _, r in df_train.iterrows()], TRAIN_PATH)
save_jsonl([to_instruction_record(r) for _, r in df_test.iterrows()], TEST_PATH)

# Reload as in lab 8.9: a list of dictionaries per split.
train_data = pd.read_json(TRAIN_PATH, lines=True).to_dict("records")
test_data = pd.read_json(TEST_PATH, lines=True).to_dict("records")
print(f"\nTrain: {len(train_data)} examples, test: {len(test_data)} examples")
print("First training example:", train_data[0])

## 5. Format the data as conversation turns

`format_io_as_turns` is the function from lab 8.9, unchanged. The Gemma 3 Keras model expects a dictionary with `"prompts"` and `"responses"` keys.

For inference, `format_question` builds the user turn in exactly the same way and then opens the model turn with `<start_of_turn>model\n`, as lab 8.9's `generate_answer` does. (Lab 8.9 puts a newline between `<end_of_turn>` and `<start_of_turn>model`; here there is no newline, because the training data built by `format_io_as_turns` has none, so the model sees the same text at inference as during training.) `extract_answer` removes the prompt and the turn markers from the generated text.

In [ ]:
# Special tokens for start/end of a turn.
SOT = "<start_of_turn>"
EOT = "<end_of_turn>"


def format_io_as_turns(
    input_text: str, output_text: str, sot: str = SOT, eot: str = EOT
) -> tuple[str, str]:
    """Formats a single input/output pair for LoRA training.

    Wraps the pair with <start_of_turn> and <end_of_turn> markers.

    Args:
      input_text: The user prompt text.
      output_text: A response to the user's query.
      sot: The token used to indicate the start of a turn.
      eot: The token used to indicate the end of a turn.

    Returns:
      formatted_q: The user turn string, e.g.
        "<start_of_turn>user\n<prompt><end_of_turn>".
      formatted_a: The model turn string, e.g.
        "<start_of_turn>model\n<label><end_of_turn>".
    """
    formatted_q = f"{sot}user\n{input_text}{eot}"
    formatted_a = f"{sot}model\n{output_text}{eot}"
    return formatted_q, formatted_a


def format_question(question: str, sot: str = SOT, eot: str = EOT) -> str:
    """Formats a question for prompting the model at inference time.

    Args:
      question: The question text.
      sot: The token used to indicate the start of a turn.
      eot: The token used to indicate the end of a turn.

    Returns:
      The user turn followed by an opened model turn, so the model continues
      with the answer.
    """
    user_turn, _ = format_io_as_turns(question.strip(), "", sot, eot)
    return f"{user_turn}{sot}model\n"


def extract_answer(raw_output: str, prompt: str) -> str:
    """Extracts the model's answer from the text returned by model.generate.

    Args:
      raw_output: The full generated text (prompt plus completion).
      prompt: The formatted prompt that was passed to model.generate.

    Returns:
      The answer text without the prompt and without turn markers.
    """
    text = raw_output
    if text.startswith(prompt):
        text = text[len(prompt):]
    else:
        marker = f"{SOT}model\n"
        idx = text.find(marker)
        if idx >= 0:
            text = text[idx + len(marker):]
    for stop in (EOT, "<eos>", SOT):
        text = text.split(stop)[0]
    return text.strip()


inputs = []  # Model prompts.
targets = []  # Expected responses.
for example in train_data:
    q, a = format_io_as_turns(example["input"], example["output"])
    inputs.append(q)
    targets.append(a)

print(inputs[0])
print(targets[0])
print("\nInference prompt example:")
print(repr(format_question(test_data[0]["input"])))

# The Gemma3 Keras models require "prompts" and "responses" keys.
data = {"prompts": inputs, "responses": targets}

## 6. Load Gemma 3 1B and test the base model

Before training, record how the **base** model answers. The same prompts are reused after fine-tuning:

* **5 fixed test prompts**: the first 5 examples of the (seeded) test split.
* **2 off-dataset prompts**: one in-syllabus topic that is not in the dataset, and one question that is not science at all (to see whether the model forces the flashcard format on everything).

Generation uses the greedy sampler so base and fine-tuned outputs are reproducible and comparable. The base model is also run on the whole test split so the automatic metrics in section 8 can compare both models.

In [ ]:
MODEL_PRESET = "gemma3_1b"
DTYPE = None  # Set to "bfloat16" if you run out of memory (see section 12).
GEN_MAX_LENGTH = 256  # Maximum total tokens (prompt plus answer) to generate.
SAMPLER = "greedy"  # Deterministic decoding for a fair comparison.

if DTYPE is None:
    model = keras_hub.models.Gemma3CausalLM.from_preset(MODEL_PRESET)
else:
    model = keras_hub.models.Gemma3CausalLM.from_preset(MODEL_PRESET, dtype=DTYPE)
model.compile(sampler=SAMPLER)
model.summary()

In [ ]:
# Check how many tokens the training examples use, so sequence_length in
# section 7 does not cut answers off.
try:
    tokenizer = model.preprocessor.tokenizer
    lengths = [len(tokenizer(q + a)) for q, a in zip(inputs, targets)]
    print(f"Tokens per training example: max {max(lengths)}, "
          f"mean {np.mean(lengths):.0f}")
except Exception as error:  # Diagnostic only, never block the notebook.
    print("Could not count tokens:", error)

In [ ]:
def generate_answer(question: str, max_length: int = GEN_MAX_LENGTH) -> str:
    """Generates the current model's answer for a single question.

    Args:
      question: The question text (without turn markers).
      max_length: Maximum total number of tokens, including the prompt.

    Returns:
      The answer text, without the prompt and turn markers.
    """
    prompt = format_question(question)
    raw_output = model.generate(prompt, max_length=max_length)
    return extract_answer(raw_output, prompt)


EVAL_EXAMPLES = test_data[:5]  # Fixed test prompts with reference answers.
OFF_DATASET_PROMPTS = [
    "What causes an earthquake?",  # In-syllabus style, not in the dataset.
    "What is the capital city of Ghana?",  # Not an Integrated Science topic.
]
SHOWCASE = [(ex["input"], ex["output"]) for ex in EVAL_EXAMPLES] + [
    (p, "") for p in OFF_DATASET_PROMPTS
]

# Base model answers for every test prompt plus the off-dataset prompts.
all_eval_prompts = [ex["input"] for ex in test_data] + OFF_DATASET_PROMPTS
base_outputs = {}
for i, prompt in enumerate(all_eval_prompts, start=1):
    base_outputs[prompt] = generate_answer(prompt)
    print(f"\rBase model: {i}/{len(all_eval_prompts)} prompts", end="")
print()

# The 7 showcase outputs, kept as a list for the comparison table.
baseline_results = [
    {"prompt": p, "expected": ref, "base": base_outputs[p]} for p, ref in SHOWCASE
]
for item in baseline_results:
    print(f"Prompt: {item['prompt']}")
    if item["expected"]:
        print(f"Expected: {item['expected'][:150]}...")
    print("Base model:")
    print(fill(item["base"], replace_whitespace=False))
    print("-" * 60)

## 7. Fine-tune with LoRA

Settings for the first run (see `RUN_IN_COLAB.md`): LoRA rank 4, learning rate 5e-5, 5 epochs, batch size 1, sequence length 256 (the cards are short, and a shorter sequence saves T4 memory). The optimizer and `compile` call follow course lab 7.5.

A callback prints generations for 3 fixed prompts after every epoch: one training question (seen), one test question (unseen, in-syllabus) and one off-syllabus question. Watch the loss go down, the answers adopt the `Category:` format, and look out for repetition or copying of training answers (signs of overfitting).

In [ ]:
LORA_RANK = 4  # Try 8 later if answers stay generic.
LEARNING_RATE = 5e-5
NUM_EPOCHS = 5
BATCH_SIZE = 1
SEQUENCE_LENGTH = 256  # Lower to 192 if you run out of memory.

# Activate LoRA: only the small low-rank matrices are trained.
model.backbone.enable_lora(rank=LORA_RANK)
model.preprocessor.sequence_length = SEQUENCE_LENGTH

optimizer = keras.optimizers.AdamW(
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    beta_1=0.9,
    beta_2=0.95,
    epsilon=1e-6,
    clipnorm=0.5,
)
# Loss and metrics use the keras_hub defaults (sparse categorical
# cross-entropy and sparse categorical accuracy on the response tokens).
model.compile(optimizer=optimizer, sampler=SAMPLER)
model.summary()

In [ ]:
MONITOR_PROMPTS = [
    train_data[0]["input"],  # Seen during training.
    test_data[0]["input"],  # Unseen, in-syllabus.
    OFF_DATASET_PROMPTS[1],  # Off-syllabus.
]


class GenerationMonitor(keras.callbacks.Callback):
    """Prints generations for the monitor prompts after each epoch."""

    def on_epoch_end(self, epoch: int, logs: dict[str, Any] | None = None):
        """Prints the model's answers for MONITOR_PROMPTS.

        Args:
          epoch: The index of the epoch that just finished.
          logs: The metric values for this epoch.
        """
        print(f"\n--- Generations after epoch {epoch + 1} ---")
        for prompt in MONITOR_PROMPTS:
            formatted_prompt = format_question(prompt)
            output = self.model.generate(formatted_prompt, max_length=GEN_MAX_LENGTH)
            print(f"Q: {prompt}")
            print(fill(extract_answer(output, formatted_prompt), replace_whitespace=False))
            print("-" * 40)


history = model.fit(
    data,
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
    callbacks=[GenerationMonitor()],
)

In [ ]:
# Plot the training loss and accuracy per epoch.
epochs_axis = range(1, len(history.history["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs_axis, history.history["loss"], marker="o")
axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
acc_key = next((k for k in history.history if "accuracy" in k), None)
if acc_key:
    axes[1].plot(epochs_axis, history.history[acc_key], marker="o", color="tab:green")
    axes[1].set_title(acc_key.replace("_", " "))
    axes[1].set_xlabel("Epoch")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_curves.png", dpi=120)
plt.show()

for epoch, loss in zip(epochs_axis, history.history["loss"]):
    print(f"Epoch {epoch}: loss {loss:.4f}")

## 8. Evaluate: base model vs fine-tuned model

First, the fine-tuned model answers the **same** prompts as the base model. The side-by-side table shows the 7 showcase prompts.

Then three automatic checks are run on **every test example**, for both models:

* **Format**: the answer starts with `Category:`.
* **Length**: the answer body (after the Category line) has 20 to 120 words, the dataset rule.
* **Token-overlap F1** (ROUGE-1 style): word overlap between the answer body and the reference body, from 0 to 1. This only measures similarity of wording, not correctness, so it complements the teacher review in section 9 but cannot replace it.

In [ ]:
finetuned_outputs = {}
for i, prompt in enumerate(all_eval_prompts, start=1):
    finetuned_outputs[prompt] = generate_answer(prompt)
    print(f"\rFine-tuned model: {i}/{len(all_eval_prompts)} prompts", end="")
print()

comparison_df = pd.DataFrame(
    [
        {
            "prompt": item["prompt"],
            "expected": item["expected"] or "(off-dataset, no reference)",
            "base": item["base"],
            "fine_tuned": finetuned_outputs[item["prompt"]],
        }
        for item in baseline_results
    ]
)
with pd.option_context("display.max_colwidth", None):
    display(
        comparison_df.style.set_properties(
            **{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"}
        )
    )
comparison_df.to_csv(f"{OUTPUT_DIR}/side_by_side.csv", index=False)

In [ ]:
def split_card(text: str) -> tuple[bool, str]:
    """Splits a flashcard into its format flag and its answer body.

    Args:
      text: A generated or reference flashcard.

    Returns:
      A tuple (has_category, body). has_category is True if the text starts
      with "Category:". body is the text after the Category line, or the whole
      text if there is no Category line.
    """
    text = text.strip()
    has_category = text.startswith("Category:")
    if has_category:
        parts = text.split("\n", 1)
        body = parts[1] if len(parts) > 1 else ""
    else:
        body = text
    return has_category, body.strip()


def word_tokens(text: str) -> list[str]:
    """Lower-cases text and returns its alphanumeric word tokens."""
    return re.findall(r"[a-z0-9]+", text.lower())


def unigram_f1(prediction: str, reference: str) -> float:
    """Computes a ROUGE-1 style F1 score from word overlap.

    Args:
      prediction: The model's answer body.
      reference: The reference answer body.

    Returns:
      The F1 score between 0 and 1.
    """
    pred, ref = word_tokens(prediction), word_tokens(reference)
    if not pred or not ref:
        return 0.0
    overlap = sum((collections.Counter(pred) & collections.Counter(ref)).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred)
    recall = overlap / len(ref)
    return 2 * precision * recall / (precision + recall)


def score_answer(answer: str, reference: str) -> dict[str, Any]:
    """Runs the automatic checks for one answer.

    Args:
      answer: The model's full answer.
      reference: The reference flashcard.

    Returns:
      A dictionary with the format flag, word count, length flag and F1.
    """
    has_category, body = split_card(answer)
    _, ref_body = split_card(reference)
    n_words = len(body.split())
    return {
        "starts_with_category": has_category,
        "answer_words": n_words,
        "length_ok_20_120": 20 <= n_words <= 120,
        "token_f1": unigram_f1(body, ref_body),
    }


rows = []
for ex in test_data:
    for system, outputs in (("base", base_outputs), ("fine_tuned", finetuned_outputs)):
        rows.append(
            {"system": system, "prompt": ex["input"],
             **score_answer(outputs[ex["input"]], ex["output"])}
        )
scores_df = pd.DataFrame(rows)
scores_df.to_csv(f"{OUTPUT_DIR}/automatic_scores.csv", index=False)

summary_df = scores_df.groupby("system").agg(
    format_ok_rate=("starts_with_category", "mean"),
    length_ok_rate=("length_ok_20_120", "mean"),
    mean_answer_words=("answer_words", "mean"),
    mean_token_f1=("token_f1", "mean"),
).round(3)
print(f"Automatic checks on {len(test_data)} test examples:")
display(summary_df)

print("\nPer-example scores for the fine-tuned model:")
display(scores_df[scores_df["system"] == "fine_tuned"].round(3))

## 9. Human evaluation sheet

Automatic scores cannot tell whether an answer is **scientifically correct** or right for a JHS student, so a JHS science teacher should score the answers from 1 (poor) to 5 (excellent):

* **accuracy_1to5**: is the science correct, with no invented facts?
* **curriculum_fit_1to5**: does it match the JHS Integrated Science syllabus and level?
* **clarity_1to5**: is it clear, short and easy for a JHS student to understand?

By default the sheet contains the fine-tuned answers **and** the base model answers for the same questions, shuffled and labelled only by `item_id`, so the teacher scores them blind. The file `human_eval_key.csv` (do not give it to the teacher) records which system wrote each item.

In [ ]:
INCLUDE_BASE_IN_SHEET = True  # Set to False to rate only the fine-tuned model.

references = {ex["input"]: ex["output"] for ex in test_data}
sheet_rows = []
for prompt in all_eval_prompts:
    systems = [("fine_tuned", finetuned_outputs)]
    if INCLUDE_BASE_IN_SHEET:
        systems.append(("base", base_outputs))
    for system, outputs in systems:
        sheet_rows.append(
            {
                "system": system,
                "prompt": prompt,
                "reference": references.get(prompt, ""),
                "model_output": outputs[prompt],
                "accuracy_1to5": "",
                "curriculum_fit_1to5": "",
                "clarity_1to5": "",
                "notes": "",
            }
        )

random.Random(SEED).shuffle(sheet_rows)
for i, row in enumerate(sheet_rows, start=1):
    row["item_id"] = i

sheet_df = pd.DataFrame(sheet_rows)
SHEET_COLUMNS = [
    "item_id", "prompt", "reference", "model_output",
    "accuracy_1to5", "curriculum_fit_1to5", "clarity_1to5", "notes",
]
SHEET_PATH = "/content/human_eval_sheet.csv"
KEY_PATH = "/content/human_eval_key.csv"
sheet_df[SHEET_COLUMNS].to_csv(SHEET_PATH, index=False, encoding="utf-8")
sheet_df[["item_id", "system", "prompt"]].to_csv(KEY_PATH, index=False)
shutil.copy(SHEET_PATH, OUTPUT_DIR)
shutil.copy(KEY_PATH, OUTPUT_DIR)
print(f"Wrote {len(sheet_df)} items to {SHEET_PATH} and the key to {KEY_PATH}.")
display(sheet_df[SHEET_COLUMNS].head())

try:
    files.download(SHEET_PATH)
    files.download(KEY_PATH)
except Exception as error:  # Downloads can fail if the browser blocks them.
    print("Automatic download failed, use the Files panel instead:", error)

After the teacher has filled in the sheet, upload it to `/content/human_eval_filled.csv` (Files panel) and run the next cell to get the average scores per system. It does nothing if the file is not there yet.

In [ ]:
FILLED_PATH = "/content/human_eval_filled.csv"
if os.path.exists(FILLED_PATH):
    filled = pd.read_csv(FILLED_PATH).merge(
        pd.read_csv(KEY_PATH)[["item_id", "system"]], on="item_id"
    )
    score_cols = ["accuracy_1to5", "curriculum_fit_1to5", "clarity_1to5"]
    for col in score_cols:
        filled[col] = pd.to_numeric(filled[col], errors="coerce")
    display(filled.groupby("system")[score_cols].agg(["mean", "count"]).round(2))
else:
    print(f"{FILLED_PATH} not found yet. Upload the filled sheet and re-run.")

## 10. Save the fine-tuned model

The LoRA weights are small (a few MB), so they are saved with `model.backbone.save_lora_weights`. To use them later, load the base model, enable LoRA with the **same rank**, and load the weights:

```python
model = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_1b")
model.backbone.enable_lora(rank=4)
model.backbone.load_lora_weights("gemma3_1b_jhs_lora_r4.lora.h5")
```

If your `keras_hub` version has no `save_lora_weights`, the cell falls back to `model.save_weights` (the full model, a few GB). Set `SAVE_FULL_PRESET = True` only if you need a standalone copy of the whole model (needed later for export and quantization for the offline device); it is several GB.

In [ ]:
SAVE_FULL_PRESET = False  # Several GB; only needed for export/quantization.

lora_path = f"{OUTPUT_DIR}/gemma3_1b_jhs_lora_r{LORA_RANK}.lora.h5"
try:
    model.backbone.save_lora_weights(lora_path)
    print(f"Saved LoRA weights to {lora_path}")
except AttributeError:
    lora_path = f"{OUTPUT_DIR}/gemma3_1b_jhs_full.weights.h5"
    model.save_weights(lora_path)
    print(f"save_lora_weights not available; saved full weights to {lora_path}")

if SAVE_FULL_PRESET:
    preset_dir = f"{OUTPUT_DIR}/gemma3_1b_jhs_preset"
    model.save_to_preset(preset_dir)
    print(f"Saved full preset to {preset_dir}")

# Record the settings and automatic scores next to the weights.
run_config = {
    "model_preset": MODEL_PRESET,
    "dtype": DTYPE,
    "lora_rank": LORA_RANK,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "sequence_length": SEQUENCE_LENGTH,
    "sampler": SAMPLER,
    "seed": SEED,
    "train_examples": len(train_data),
    "test_examples": len(test_data),
    "final_loss": float(history.history["loss"][-1]),
    "automatic_scores": summary_df.reset_index().to_dict("records"),
}
with open(f"{OUTPUT_DIR}/run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2, default=str)

if USE_DRIVE:
    shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
    print(f"Copied outputs to {DRIVE_OUTPUT_DIR}")

print("Saved files:", sorted(os.listdir(OUTPUT_DIR)))

# Zip the outputs for download (skipped when the multi-GB preset was saved).
if SAVE_FULL_PRESET:
    print("Full preset saved: copy it via Drive (USE_DRIVE = True) instead.")
else:
    zip_path = shutil.make_archive("/content/jhs_capstone_outputs", "zip", OUTPUT_DIR)
    try:
        files.download(zip_path)
    except Exception as error:  # Downloads can fail if the browser blocks them.
        print("Automatic download failed, use the Files panel instead:", error)

## 11. Iteration and reflection

### Iteration

Use the results above to plan your next run. Write your answers in your own words:

* Which topics or question types did the fine-tuned model get wrong or answer vaguely? (Look at the side-by-side table, the per-example scores and the teacher's notes.)
* What will you change in the data (for example, more cards or more varied phrasing for weak topics in `data/jhs_flashcards.json`, then re-export with `scripts/export_jsonl.py`)?
* What will you change in training (epochs, learning rate, LoRA rank 4 vs 8, sequence length), and why?
* After the next run, what changed in the loss, the automatic scores and the teacher scores?

_Your notes:_

### Reflection

* What did the model learn well, and where does it still fail?
* What risks remain for students (for example, confident but wrong answers), and how would you reduce them before classroom use?
* What did you learn about fine-tuning, evaluation and working with a teacher?
* What would you do next (more subjects, other languages, the offline device)?

_Your reflection:_

## 12. If you run out of GPU memory

If training stops with an out-of-memory error (`RESOURCE_EXHAUSTED` or similar):

1. Set `SEQUENCE_LENGTH = 192` in section 7 (the cards are short, so this rarely cuts answers off; check the token count printed in section 6). Keep `BATCH_SIZE = 1` and `LORA_RANK = 4`.
2. If that is not enough, set `DTYPE = "bfloat16"` in section 6 to load the model in half precision (course lab 7.5).
3. Then use **Runtime > Restart session** and **Run all** again (memory is only freed by a restart).